<a href="https://colab.research.google.com/github/goutham3010/Hospital-Readmission-Prediction-System/blob/main/07_Explainable_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ============================================================
# STEP 07: EXPLAINABLE AI (XAI)
# Hospital Readmission Prediction System
# ============================================================


**Objective:** Explain how our tuned Random Forest model makes predictions so doctors can trust and understand the model.

In healthcare, we cannot just use a "black box" model. If a model says a patient has a 95% chance of being readmitted, doctors need to know:
1. **Global Explainability:** Which patient features are generally most important across all patients?
2. **Local Explainability:** Why did the model predict high risk for a *specific* patient?

Let's dive in!

### 1. Import Libraries
First, let's import the standard libraries for data handling and plotting.

In [ ]:
# 1. Import libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

print("Libraries loaded successfully!")

### 2. Load the Dataset and Saved Best Model
Let's load our hospital dataset and the best model we saved in previous steps.

In [ ]:
# 2. Load dataset (works in Google Colab or locally)
possible_paths = [
    "hospital_readmission_dataset.csv",
    "hospital_readmission_dataset (1).csv",
    "/content/hospital_readmission_dataset.csv"
]
data_path = next((p for p in possible_paths if os.path.exists(p)), "hospital_readmission_dataset.csv")
df = pd.read_csv(data_path)
print(f"Loaded dataset from: {data_path}")
print("Dataset shape:", df.shape)
display(df.head(3))

# Load our saved best model
model_paths = ["best_readmission_model.joblib", "best_readmission_model.pkl", "/content/best_readmission_model.pkl"]
model_path = next((p for p in model_paths if os.path.exists(p)), None)

if model_path:
    best_model = joblib.load(model_path)
    print(f"Loaded saved model from: {model_path}")
else:
    print("Saved model not found, please train and save best_model first!")

### 3. Prepare Test Data
We drop ID and leak columns, then split into train and test sets exactly like in Notebook 03 and 04.

In [ ]:
# 3. Prepare features (X) and label (y)
drop_cols = ["patient_id", "admission_date", "readmission_risk_score"]
df_clean = df.drop(columns=[c for c in drop_cols if c in df.columns])

X = df_clean.drop(columns=["label"])
y = df_clean["label"]

# 80% train, 20% test with stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Test set size:", X_test.shape)
print("Features count:", X.shape[1])

### 4. Global Explainability: Feature Importance from Random Forest
Random Forest calculates **Gini Importance** (Mean Decrease in Impurity).
Let's extract the feature names from our pipeline and see which ones got the highest importance score!

In [ ]:
# 4. Extract feature importances from the pipeline
preprocessor = best_model.named_steps["preprocessor"]
rf_classifier = best_model.named_steps["model"]

# Get all feature names after one-hot encoding
feature_names = preprocessor.get_feature_names_out()

# Make the feature names look clean and readable
clean_names = [
    name.replace("num__", "").replace("cat__", "").replace("_", " ").title()
    for name in feature_names
]

# Create a nice DataFrame of importances
feature_importance_df = pd.DataFrame({
    "Feature": clean_names,
    "Importance": rf_classifier.feature_importances_
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

print("Top 10 Most Important Features:")
display(feature_importance_df.head(10))

### 5. Plotting Feature Importance
Let's visualize the top 10 features using a horizontal bar chart so it's easy to read.

In [ ]:
# 5. Plot the top 10 features
top10 = feature_importance_df.head(10).iloc[::-1]  # reverse order so highest is on top

plt.figure(figsize=(10, 6))
plt.barh(top10["Feature"], top10["Importance"], color="#3b82f6", edgecolor="#1d4ed8")
plt.title("Top 10 Clinical Features Driving Hospital Readmission (XAI)", fontsize=13, fontweight="bold")
plt.xlabel("Feature Importance Score (Gini)", fontsize=11)
plt.ylabel("Clinical Feature", fontsize=11)
plt.grid(axis="x", linestyle="--", alpha=0.5)

# Add labels to each bar
for i, v in enumerate(top10["Importance"]):
    plt.text(v + 0.002, i, f"{v*100:.1f}%", va="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()

#### 💡 Student Observations on Global Importance:
1. **Age (~15.3%)** is by far the biggest factor. Older patients naturally have a harder time recovering and face higher risks.
2. **Medications Count (~9.6%)** is #2! When patients take a lot of medicines (polypharmacy), they are much more prone to side effects or forgetting pills.
3. **Length of Stay (~7.9%) & Prev Readmissions (~7.7%)**: Patients who stayed longer in the hospital or have been admitted before have a much higher likelihood of coming back.
4. **Comorbidities (~7.4%)**: Having multiple underlying conditions (like Diabetes + Heart issues) increases risk significantly.

### 6. Permutation Importance Check
What happens if we randomly shuffle a column and see how much the accuracy drops?  
This gives another way to verify feature importance without relying only on tree splits.

In [ ]:
# 6. Permutation Importance on a sample of test data
print("Running Permutation Importance (this checks how much score drops when a feature is shuffled)...")
perm_result = permutation_importance(
    best_model, X_test.head(400), y_test.head(400), n_repeats=5, random_state=42, scoring="f1"
)

perm_df = pd.DataFrame({
    "Feature": X_test.columns,
    "F1_Score_Drop": perm_result.importances_mean
}).sort_values(by="F1_Score_Drop", ascending=False).reset_index(drop=True)

print("Permutation Importance Results:")
display(perm_df.head(6))

### 7. Local Explainability: Explaining Single Patient Predictions
Now let's zoom in on **individual patients**.
Let's write a simple helper function `explain_patient(patient_data)` that calculates the patient's risk and shows which specific factors made the model give that score.

In [ ]:
# 7. Helper function to explain single patient predictions
def explain_patient(patient_df):
    # Get predicted probability from our pipeline
    risk_prob = best_model.predict_proba(patient_df)[0, 1]

    # Assign risk tier
    if risk_prob >= 0.70:
        tier = "🔴 HIGH RISK"
        advice = "Action: Nurse home visit within 48 hours & medication review!"
    elif risk_prob >= 0.40:
        tier = "🟡 MEDIUM RISK"
        advice = "Action: Telephone check-in within 3 days & follow-up appointment in 1 week."
    else:
        tier = "🟢 LOW RISK"
        advice = "Action: Standard discharge instructions & 30-day routine check-up."

    print("=" * 60)
    print("PATIENT READMISSION RISK REPORT")
    print("=" * 60)
    print(f"Predicted Readmission Risk : {risk_prob*100:.2f}%")
    print(f"Clinical Tier              : {tier}")
    print(f"Recommended Doctor Action  : {advice}")
    print("-" * 60)
    print("Key Contributing Factors for this Patient:")

    row = patient_df.iloc[0]
    if row["age"] >= 65:
        print(f"  [!] Senior Age ({row['age']} yrs) increases vulnerability to complications.")
    if row["comorbidities_count"] >= 4:
        print(f"  [!] High Comorbidities ({row['comorbidities_count']} conditions) creates high disease burden.")
    if row["length_of_stay"] >= 7:
        print(f"  [!] Long Hospital Stay ({row['length_of_stay']} days) indicates severe acute illness.")
    if row["medications_count"] >= 8:
        print(f"  [!] Polypharmacy ({row['medications_count']} meds) puts patient at high risk for drug interactions.")
    if row["prev_readmissions"] >= 2:
        print(f"  [!] History of Readmission ({row['prev_readmissions']} past admissions) shows chronic instability.")
    if row["discharge_disposition"] in ["Skilled Nursing", "Home Health"]:
        print(f"  [!] Discharge to {row['discharge_disposition']} means patient still needs active medical assistance.")
    print("=" * 60)

# Let's test it on a sample patient from the test set!
sample_patient = X_test.iloc[[0]]
explain_patient(sample_patient)

### 8. Case Study Comparison: High Risk vs Low Risk Patient
Let's compare two different patients side by side to see how the model behaves on different patient profiles.

In [ ]:
# Case 1: An older, high-risk patient profile
high_risk_patient = pd.DataFrame([{
    "season": "Winter",
    "age": 78,
    "gender": "Female",
    "region": "South",
    "primary_diagnosis": "Heart Failure",
    "comorbidities_count": 5,
    "length_of_stay": 9,
    "treatment_type": "Medical",
    "medications_count": 12,
    "followup_visits_last_year": 4,
    "prev_readmissions": 3,
    "insurance_type": "Medicare",
    "discharge_disposition": "Skilled Nursing"
}])

# Case 2: A younger, healthy profile
low_risk_patient = pd.DataFrame([{
    "season": "Summer",
    "age": 32,
    "gender": "Male",
    "region": "North",
    "primary_diagnosis": "Appendicitis",
    "comorbidities_count": 0,
    "length_of_stay": 2,
    "treatment_type": "Surgical",
    "medications_count": 2,
    "followup_visits_last_year": 1,
    "prev_readmissions": 0,
    "insurance_type": "Private",
    "discharge_disposition": "Home"
}])

print("CASE STUDY 1 (HIGH RISK PATIENT):")
explain_patient(high_risk_patient)

print("\nCASE STUDY 2 (LOW RISK PATIENT):")
explain_patient(low_risk_patient)

### 9. Conclusion & Project Summary

In this Explainable AI notebook, we accomplished:
1. **Disproved the "Black Box" Myth:** We proved that our Random Forest model isn't making mysterious guesses. It relies on clinically sound factors like patient age, multiple medications, length of stay, and chronic illnesses.
2. **Global Insights:** Age (15.3%), Medications Count (9.6%), and Length of Stay (7.9%) are the top 3 drivers of hospital readmissions across our entire patient dataset.
3. **Local Actionable Explanations:** We created a patient-level report that gives hospital discharge teams actionable next steps (e.g. nurse home visits vs standard discharge) tailored to each patient's exact risk profile.

